In [3]:
# Importing the libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [4]:
# ── 1. Load Dataset ───────────────────────────────

df = pd.read_csv("datasets/admission_data.csv")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (500, 8)
   GRE Score  TOEFL Score  University Rating  SOP  LOR   CGPA  Research  \
0        337          118                  4  4.5   4.5  9.65         1   
1        324          107                  4  4.0   4.5  8.87         1   
2        316          104                  3  3.0   3.5  8.00         1   
3        322          110                  3  3.5   2.5  8.67         1   
4        314          103                  2  2.0   3.0  8.21         0   

   Chance of Admit   
0              0.92  
1              0.76  
2              0.72  
3              0.80  
4              0.65  


In [5]:
# ── 2. Select Features & Target ───────────────────────────────────────────
# Using only GRE Score and CGPA as inputs (as it is the requirement of the task)
X = df[["GRE Score", "CGPA"]].values
y = df["Chance of Admit "].values      # trailing space in original column name

In [6]:
# ── 3. Normalize Features ─────────────────────────────────────────────────
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [7]:
# ── 4. Train / Test Split ─────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"\nTrain samples: {len(X_train)}, Test samples: {len(X_test)}")


Train samples: 400, Test samples: 100


In [8]:
# ── 5. Build Neural Network ───────────────────────────────────────────────
model = keras.Sequential([
    layers.Input(shape=(2,)),          # GRE Score, CGPA
    layers.Dense(8, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
], name="admission_predictor")

model.summary()

Model: "admission_predictor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105 (420.00 B)

 Trainable params: 105 (420.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
# ── 6. Compile ────────────────────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="mse",
    metrics=["mae"]
)

In [10]:
# ── 7. Train ──────────────────────────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=16,
    validation_split=0.1,
    verbose=1
)

Epoch 1/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0557 - mae: 0.2013 - val_loss: 0.0326 - val_mae: 0.1516
Epoch 2/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0158 - mae: 0.0995 - val_loss: 0.0153 - val_mae: 0.1051
Epoch 3/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0132 - mae: 0.0895 - val_loss: 0.0124 - val_mae: 0.0959
Epoch 4/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0089 - mae: 0.0750 - val_loss: 0.0057 - val_mae: 0.0664
Epoch 5/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0049 - mae: 0.0522 - val_loss: 0.0040 - val_mae: 0.0501
Epoch 6/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0048 - mae: 0.0527 - val_loss: 0.0034 - val_mae: 0.0475
Epoch 7/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0046 - mae: 0.0516 - val_loss: 0.0032 - val_mae: 0.0459
Epoch 8/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0044 - mae: 0.0492 - val_loss: 0.0034 - val_mae: 0.0473
Epoch 9/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.004

In [11]:
# ── 8. Evaluate ───────────────────────────────────────────────────────────
loss, mae = model.evaluate(X_test, y_test, verbose=0)
y_pred = model.predict(X_test).flatten()
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n── Test Results ──────────────────────────")
print(f"  MSE  : {loss:.4f}")
print(f"  MAE  : {mae:.4f}")
print(f"  RMSE : {rmse:.4f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 

── Test Results ──────────────────────────
  MSE  : 0.0044
  MAE  : 0.0493
  RMSE : 0.0665


In [12]:
# ── 9. Sample Predictions ─────────────────────────────────────────────────
print(f"\n── Predictions vs Actual (first 10 test samples) ──")
print(f"{'GRE':>6}  {'CGPA':>6}  {'Actual':>8}  {'Predicted':>10}  {'Error':>8}")
print("-" * 50)

X_test_orig = scaler.inverse_transform(X_test)
for i in range(min(10, len(X_test))):
    gre  = X_test_orig[i][0]
    cgpa = X_test_orig[i][1]
    print(f"{gre:>6.0f}  {cgpa:>6.2f}  {y_test[i]:>8.2f}  {y_pred[i]:>10.3f}  {abs(y_pred[i]-y_test[i]):>8.3f}")



── Predictions vs Actual (first 10 test samples) ──
   GRE    CGPA    Actual   Predicted     Error
--------------------------------------------------
   334    9.54      0.93       0.923     0.007
   314    9.04      0.84       0.760     0.080
   315    7.65      0.39       0.571     0.181
   312    8.69      0.77       0.709     0.061
   326    9.05      0.74       0.798     0.058
   329    9.23      0.89       0.858     0.032
   290    7.56      0.47       0.470     0.000
   301    8.47      0.57       0.632     0.062
   318    9.22      0.68       0.812     0.132
   320    8.86      0.82       0.758     0.062


In [13]:
# ── 10. Predict for New Input ─────────────────────────────────────────────
def predict_admission(gre_score, cgpa):
    x_new = scaler.transform([[gre_score, cgpa]])
    prob  = model.predict(x_new, verbose=0)[0][0]
    print(f"\nGRE: {gre_score}  CGPA: {cgpa:.2f}  →  Admission Chance: {prob*100:.1f}%")
    return prob

predict_admission(320, 8.5)
predict_admission(310, 7.9)
predict_admission(335, 9.5)



GRE: 320  CGPA: 8.50  →  Admission Chance: 71.4%

GRE: 310  CGPA: 7.90  →  Admission Chance: 58.7%

GRE: 335  CGPA: 9.50  →  Admission Chance: 92.2%


np.float32(0.9224192)